In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import os

from ipywidgets import VBox, Dropdown, FloatText, IntText, Button, Output, Layout
from IPython.display import display, clear_output


In [ ]:

# ============================================================
# 1. Load train/test
# ============================================================
train_path = "./data/split/train/train.csv"
test_path  = "./data/split/test/test.csv"

df_train = pd.read_csv(train_path)
df_test  = pd.read_csv(test_path)



In [ ]:

# ============================================================
# 2. Load address → cluster mapping (mode của train)
# ============================================================
map_path = "./model/model6/address_cluster_map.csv"
if not os.path.isfile(map_path):
    raise FileNotFoundError("Thiếu file address_cluster_map.csv")

address_cluster_map = pd.read_csv(map_path)
addr_to_cluster = dict(zip(address_cluster_map["address"], address_cluster_map["cluster_mode"]))

# thêm cluster vào train (để vẽ scatter)
df_train["cluster"] = df_train["address"].map(addr_to_cluster)


In [ ]:

# ============================================================
# 3. Load pre-trained model cho từng cluster
# ============================================================
models = {}
for cid in [0, 1, 2]:
    pt_path = f"./model/model6/cluster_{cid}.pt"
    if not os.path.isfile(pt_path):
        raise FileNotFoundError(f"Thiếu file {pt_path}")

    m = nn.Linear(2, 1)
    m.load_state_dict(torch.load(pt_path, map_location="cpu"))
    m.eval()
    models[cid] = m

print("✔ Loaded 3 models & address mapping.")


In [ ]:

# ============================================================
# 4. UI Components
# ============================================================
address_list = sorted(address_cluster_map["address"].unique().tolist())

address_w = Dropdown(options=address_list, description="Address:", layout=Layout(width="350px"))
area_w    = FloatText(value=50.0, description="Area:", layout=Layout(width="250px"))
bed_w     = IntText(value=2, description="Bedrooms:", layout=Layout(width="250px"))
run_btn   = Button(description="Predict", button_style="primary", layout=Layout(width="150px"))
out       = Output()


In [ ]:


# ============================================================
# 5. Prediction Handler
# ============================================================
@run_btn.on_click
def _run(_):
    with out:
        clear_output(wait=True)

        sel_addr = address_w.value
        cluster_id = addr_to_cluster.get(sel_addr, 1)

        x = np.array([[area_w.value, bed_w.value]], dtype=np.float32)
        x_t = torch.tensor(x)

        # predict
        with torch.no_grad():
            y_pred = models[cluster_id](x_t).item()

        print(f"Address: {sel_addr} → Cluster {cluster_id}")
        print(f"Predicted price: {y_pred:,.2f} tỷ VNĐ")

        # =====================================================
        # visualization
        # =====================================================
        clust_df = df_train[df_train["cluster"] == cluster_id]

        if clust_df.empty:
            print("Không có dữ liệu train cho cluster này.")
            return

        fig, ax = plt.subplots(figsize=(10, 6))
        ax.scatter(clust_df["area"], clust_df["price"], c="blue", alpha=0.5)

        # Draw linear model line
        area_range = np.linspace(clust_df["area"].min(), clust_df["area"].max(), 100)
        X_line = torch.tensor(
            np.column_stack([area_range, np.full_like(area_range, bed_w.value)]),
            dtype=torch.float32
        )
        with torch.no_grad():
            y_line = models[cluster_id](X_line).numpy().ravel()

        ax.plot(area_range, y_line, "r-", label="Model line")
        ax.scatter([area_w.value], [y_pred], c="yellow", s=150, edgecolors="black")

        ax.set_xlabel("Area (m²)")
        ax.set_ylabel("Price (tỷ VNĐ)")
        ax.set_title(f"Cluster {cluster_id}: Price vs Area")
        ax.grid(True)
        plt.show()



In [ ]:

# ============================================================
# 6. Display UI
# ============================================================
display(VBox([address_w, area_w, bed_w, run_btn, out]))